# 01 · Data overview and cleaning

Goal: understand the bookings data, find its traps and save a clean file for the analysis and the Tableau dashboard.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/hotel_bookings.csv")
print(df.shape)
df.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 1.1 Structure
How many rows and columns? Which columns are numbers, which are text? (`df.info()`)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

## 1.2 Missing values
Which columns have missing values, and how many?

In [3]:
df.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

Missing values: children (4) and country (488) are truly missing; agent (13.7%) and company (94.3%) are NaN when there is no agent/company, so they carry meaning and are not errors.
agent and company are IDs stored as floats because of NaN: treat them as categories, not numbers.

## 1.3 Duplicates
How many rows are exact duplicates? Are they errors, or could they be real bookings?

In [4]:
n_dup = df.duplicated().sum()
print(n_dup, f"{n_dup / len(df): .1%}")

31994  26.8%


In [5]:
df[df.duplicated(keep=False)].head(6)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
21,Resort Hotel,0,72,2015,July,27,1,2,4,2,...,No Deposit,250.0,NaN,0,Transient,84.67,0,1,Check-Out,2015-07-07
22,Resort Hotel,0,72,2015,July,27,1,2,4,2,...,No Deposit,250.0,NaN,0,Transient,84.67,0,1,Check-Out,2015-07-07
39,Resort Hotel,0,70,2015,July,27,2,2,3,2,...,No Deposit,250.0,NaN,0,Transient,137.00,0,1,Check-Out,2015-07-07
43,Resort Hotel,0,70,2015,July,27,2,2,3,2,...,No Deposit,250.0,NaN,0,Transient,137.00,0,1,Check-Out,2015-07-07


In [6]:
n_dup = df.duplicated().sum()
print(f"Duplicate rows: {n_dup} ({n_dup / len(df):.1%})")

df_dedup = df.drop_duplicates()
print(f"Cancellation rate, all rows: {df['is_canceled'].mean():.1%}")
print(f"Cancellation rate, no duplicates: {df_dedup['is_canceled'].mean():.1%}")

Duplicate rows: 31994 (26.8%)
Cancellation rate, all rows: 37.0%
Cancellation rate, no duplicates: 27.5%


In [7]:
dup_rows = df[df.duplicated(keep=False)]
unique_rows = df[~df.duplicated(keep=False)]

print(f"Cancellation rate, duplicated rows: {dup_rows['is_canceled'].mean():.1%}")
print(f"Cancellation rate, unique rows:     {unique_rows['is_canceled'].mean():.1%}")
print()
print(dup_rows["deposit_type"].value_counts(normalize=True).round(3))
print()
print(dup_rows["market_segment"].value_counts(normalize=True).round(3).head())

Cancellation rate, duplicated rows: 58.4%
Cancellation rate, unique rows:     26.2%

deposit_type
No Deposit    0.642
Non Refund    0.356
Refundable    0.002
Name: proportion, dtype: float64

market_segment
Groups           0.416
Offline TA/TO    0.304
Online TA        0.206
Corporate        0.037
Direct           0.035
Name: proportion, dtype: float64


In [8]:
df["is_duplicate"] = df.duplicated(keep=False).astype(int)
df["is_duplicate"].sum()

np.int64(40165)

31,994 rows (26.8%) are exact copies of another row; in total, 40,165 rows belong to 8,171 groups of identical rows. The data is anonymised (no booking ID), so identical rows can be separate rooms in the same group booking. These rows are mostly Groups and Offline TA/TO (72%) and are cancelled far more often (58.4% vs 26.2%). Dropping the copies would cut the cancellation rate from 37.0% to 27.5% and remove the largest cancellations, so all rows are kept and flagged with is_duplicate (keep=False).

## 1.4 Impossible values
- Bookings with 0 adults, 0 children and 0 babies
- Negative or extreme `adr` (average daily rate)

In [9]:
no_guests = (df['adults'] + df['children'].fillna(0) + df['babies']) == 0
print(no_guests.sum(), 'bookings with no guests')
df.loc[no_guests, ['hotel', 'is_canceled', 'market_segment', 'adr', 'stays_in_week_nights']].head()

180 bookings with no guests


,hotel,is_canceled,market_segment,adr,stays_in_week_nights
2224,Resort Hotel,0,Corporate,0.0,3
2409,Resort Hotel,0,Corporate,0.0,0
3181,Resort Hotel,0,Groups,0.0,2
3684,Resort Hotel,0,Groups,0.0,4
3708,Resort Hotel,0,Groups,0.0,4


180 bookings (0.15%) have no adults, children or babies, mostly Groups and Corporate with adr = 0. A stay with no guests is not possible, so they are removed (likely held rooms or entry errors).

In [10]:
print(df["adr"].describe())
print()
print((df["adr"] < 0).sum(), "rows with negative adr")
print((df["adr"] == 0).sum(), "rows with adr = 0")
df.sort_values("adr", ascending=False)[["hotel", "adr", "adults", "stays_in_week_nights", "is_canceled"]].head(3)

count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

1 rows with negative adr
1959 rows with adr = 0


,hotel,adr,adults,stays_in_week_nights,is_canceled
48515,City Hotel,5400.0,2,1,1
111403,City Hotel,510.0,1,1,0
15083,Resort Hotel,508.0,2,1,0


In [11]:
df.loc[df["adr"] == 0, "market_segment"].value_counts()

market_segment
Complementary    680
Online TA        367
Offline TA/TO    332
Groups           252
Direct           240
Corporate         82
Aviation           6
Name: count, dtype: int64

adr: 1 negative value (−6.38) and 1 extreme value (5,400 for a 1-night city stay, 10x the next highest at 510) are removed as errors. 1,959 bookings have adr = 0: 680 are Complementary (free stays by definition), the rest may be package deals or entry errors. They are kept for counts and cancellation rates but excluded from ADR averages.

In [13]:
# Copy of the original dataset with applied modifications
df_clean = df.copy()

# 1.4 - remove impossible rows
no_guests = (df_clean["adults"] + df_clean["children"].fillna(0) + df_clean["babies"]) == 0
bad_adr = (df_clean["adr"] < 0) | (df_clean["adr"] > 5000)
df_clean = df_clean[~no_guests & ~bad_adr]

# 1.2 - missing values
df_clean["children"] = df_clean["children"].fillna(0).astype(int)
df_clean["country"] = df_clean["country"].fillna("Unknown")
df_clean["agent"] = df_clean["agent"].fillna(0).astype(int)        # 0 = no agent
df_clean["has_company"] = df_clean["company"].notna().astype(int)  # 1 = corporate booking
df_clean = df_clean.drop(columns="company")

# 1.1 - types
df_clean["reservation_status_date"] = pd.to_datetime(df_clean["reservation_status_date"])

print(len(df), "->", len(df_clean), "rows")
print(df_clean.isna().sum().sum(), "missing values")

119390 -> 119208 rows
0 missing values


## 1.5 New columns
- `arrival_date` (a real date built from year, month and day)
- `total_nights` (weekend + week nights)
- `revenue` (`adr` × `total_nights`)

In [20]:
# Year month day
df_clean["arrival_date"] = pd.to_datetime(
    df_clean["arrival_date_year"].astype(str) + "-"
    + df_clean["arrival_date_month"] + "-"
    + df_clean["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

# Total nights of the stay
df_clean["total_nights"] = df_clean["stays_in_weekend_nights"] + df_clean["stays_in_week_nights"]

# Revenue of booking
df_clean['revenue'] = df_clean['adr'] * df_clean['total_nights']


df_clean[['arrival_date','total_nights','adr','revenue']].head()

,arrival_date,total_nights,adr,revenue
0,2015-07-01,0,0.0,0.0
1,2015-07-01,0,0.0,0.0
2,2015-07-01,1,75.0,75.0
3,2015-07-01,1,75.0,75.0
4,2015-07-01,2,98.0,196.0


In [21]:
print(df_clean["arrival_date"].min(), "->", df_clean["arrival_date"].max())
print((df_clean["total_nights"] == 0).sum(), "bookings with 0 nights")
print(df_clean["total_nights"].describe())

2015-07-01 00:00:00 -> 2017-08-31 00:00:00
645 bookings with 0 nights
count    119208.000000
mean          3.426213
std           2.540633
min           0.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          69.000000
Name: total_nights, dtype: float64


Arrivals from 1 July 2015 to 31 August 2017 (only 2016 is a full year: compare months, not yearly totals). Median stay 3 nights; 645 bookings (0.5%) have 0 nights (likely day use) and are kept, with revenue 0. The longest stay (69 nights) is rare but plausible.

## 1.6 Columns we must not use to predict cancellations
Compare `reservation_status` with `is_canceled`. What do you notice?

In [22]:
pd.crosstab(df_clean["reservation_status"], df_clean["is_canceled"])

is_canceled,0,1
reservation_status,,
Canceled,0,42992
Check-Out,75010,0
No-Show,0,1206


Data leakage: reservation_status maps 1:1 to is_canceled (Canceled and No-Show = 1, Check-Out = 0) and is only known after the stay. reservation_status_date and assigned_room_type are also set after booking time. They are kept for reporting but excluded from the prediction model. No-shows (1,206) are counted as cancellations.

## 1.7 Save the clean file
Save to `../data/processed/hotel_bookings_clean.csv`.

In [23]:
df_clean.to_csv("../data/processed/hotel_bookings_clean.csv", index=False)
print(df_clean.shape)

(119208, 36)
